<a href="https://colab.research.google.com/github/KaveeshwaraBandara/EN4554-A2-CNNs-CIFAR10/blob/main/220059J_a02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EN4554 Deep Learning for Vision — Assignment 2: CNNs

**Name:** Imanchana Kaveeshwara Bandara
**Index number:** `220059j`
**GitHub repository:** `https://github.com/KaveeshwaraBandara/EN4554-A2-CNNs-CIFAR10.git`

**Use of AI tools.** Claude (Anthropic) was used to help structure the notebook and write parts of the code. I reviewed, ran and understood all code, and the interpretations of the results are my own.


## 0. Setup

In [ ]:
# ===================== Configuration =====================
SMOKE_TEST = False   # True: tiny data + 1 epoch per run, to check the whole pipeline in a few minutes
RUN_TASK4  = True    # Task 4 is optional
SEED = 42
BS   = 128           # batch size used everywhere

EPOCHS_1C = 20       # Task 1(c)  manual MLP
EPOCHS_2C = 15       # Task 2(c)  9 runs (3 optimizers x 3 learning rates)
EPOCHS_3A = 60       # Task 3(a)  7 runs on 5k images (cheap)
EPOCHS_3B = 30       # Task 3(b)  6 runs, plain/residual depth 8/20/56 (the expensive part)
EPOCHS_4  = 20       # Task 4     limit given in the assignment

In [ ]:
import os, json, time, math, copy, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F, torchvision

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = (device == 'cuda')                 # mixed precision (fp16) on the GPU only, used from Task 2(c) on
torch.backends.cudnn.benchmark = True    # faster convolutions (tiny run-to-run non-determinism)
set_seed()
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '', '| torch', torch.__version__)

In [ ]:
# ---- Results cache on Google Drive (survives disconnects) ----
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS = '/content/drive/MyDrive/EN4554_A2/results'
except ImportError:                      # running on a laptop
    RESULTS = './results'
if SMOKE_TEST:
    RESULTS += '_smoke'                  # never mix smoke-test results with real ones
os.makedirs(RESULTS, exist_ok=True)

def save_json(name, obj):
    with open(f'{RESULTS}/{name}.json', 'w') as f: json.dump(obj, f)

def load_json(name):
    p = f'{RESULTS}/{name}.json'
    return json.load(open(p)) if os.path.exists(p) else None

def cached(name, fn):
    # Run fn() once and store its result dict; afterwards load it from the cache.
    r = load_json(name)
    if r is not None:
        print(f'[cache] {name}')
        return r
    t0 = time.time(); r = fn(); r['time_s'] = time.time() - t0
    save_json(name, r)
    print(f'[done]  {name}  ({r["time_s"]/60:.1f} min)')
    return r

def savefig(name):
    plt.savefig(f'{RESULTS}/{name}.png', dpi=150, bbox_inches='tight')

print('results ->', RESULTS)

In [ ]:
# ---- CIFAR-10: 45k train / 5k validation, normalized with TRAINING statistics ----
tr_raw = torchvision.datasets.CIFAR10('./data', train=True,  download=True)
te_raw = torchvision.datasets.CIFAR10('./data', train=False, download=True)
X = torch.tensor(np.asarray(tr_raw.data)).permute(0, 3, 1, 2).float() / 255.   # (50000, 3, 32, 32)
Y = torch.tensor(tr_raw.targets)

g = torch.Generator().manual_seed(SEED)          # fixed split
perm = torch.randperm(50000, generator=g)
X_tr, Y_tr = X[perm[:45000]], Y[perm[:45000]]
X_va, Y_va = X[perm[45000:]], Y[perm[45000:]]

mean = X_tr.mean(dim=(0, 2, 3), keepdim=True)    # per-channel stats from the 45k training images only
std  = X_tr.std(dim=(0, 2, 3), keepdim=True)
norm = lambda x: (x - mean) / std

X_tr, X_va = norm(X_tr).to(device), norm(X_va).to(device)   # the whole dataset lives on the GPU (~600 MB)
Y_tr, Y_va = Y_tr.to(device), Y_va.to(device)

# Test set: same TRAIN statistics; kept on the CPU and used exactly once, in Task 4
X_te = norm(torch.tensor(np.asarray(te_raw.data)).permute(0, 3, 1, 2).float() / 255.)
Y_te = torch.tensor(te_raw.targets)
del X, Y

if SMOKE_TEST:
    X_tr, Y_tr, X_va, Y_va = X_tr[:2048], Y_tr[:2048], X_va[:512], Y_va[:512]
    X_te, Y_te = X_te[:1000], Y_te[:1000]
    EPOCHS_1C = EPOCHS_2C = EPOCHS_3A = EPOCHS_3B = EPOCHS_4 = 1

CLASSES = tr_raw.classes
print('train', tuple(X_tr.shape), '| val', tuple(X_va.shape), '| test', tuple(X_te.shape))
print('channel mean', mean.flatten().numpy().round(4), '| std', std.flatten().numpy().round(4))
print('val images per class', torch.bincount(Y_va.cpu(), minlength=10).tolist())

def batches(X, Y, bs=BS, shuffle=True):
    # Mini-batches straight from GPU memory (much faster than a DataLoader on Colab)
    idx = torch.randperm(len(X)).to(X.device) if shuffle else torch.arange(len(X), device=X.device)
    for i in range(0, len(X), bs):
        j = idx[i:i + bs]
        yield X[j], Y[j]

## Task 1: Backpropagation

### 1(a) Layers with explicit forward/backward (no autograd)

For a linear layer $Y = XW^\top + b$ with $X\in\mathbb R^{N\times D_{in}}$, $W\in\mathbb R^{D_{out}\times D_{in}}$ and upstream gradient $\delta = \partial L/\partial Y$:

$$\frac{\partial L}{\partial W} = \delta^\top X,\qquad \frac{\partial L}{\partial b} = \sum_{n}\delta_n,\qquad \frac{\partial L}{\partial X} = \delta\,W.$$

ReLU: $\partial L/\partial x = \delta\odot\mathbb 1[x>0]$. Sigmoid: $\partial L/\partial x = \delta\odot s(1-s)$.

Softmax + cross-entropy (mean over the batch), with $p=\mathrm{softmax}(z)$ and one-hot targets $y$:

$$L = \frac1N\sum_n\Big(-z_{n,y_n} + \log\sum_j e^{z_{nj}}\Big),\qquad \frac{\partial L}{\partial z} = \frac{1}{N}\,(p - y).$$

The max logit is subtracted before exponentiating (log-sum-exp trick) so that $e^{z}$ cannot overflow.

In [ ]:
class Linear:
    def __init__(self, din, dout, init='kaiming', dtype=torch.float32):
        # Kaiming (He) init keeps the activation variance constant through ReLU layers; Xavier suits sigmoid/tanh
        std = math.sqrt(2.0 / din) if init == 'kaiming' else math.sqrt(1.0 / din)
        self.W = torch.randn(dout, din, dtype=dtype, device=device) * std
        self.b = torch.zeros(dout, dtype=dtype, device=device)
    def forward(self, x):
        self.x = x                          # cache the input for the backward pass
        return x @ self.W.T + self.b
    def backward(self, dout):
        self.dW = dout.T @ self.x           # (Dout,N) @ (N,Din)  -> sums over the batch
        self.db = dout.sum(0)               # b is shared by every sample -> sum
        return dout @ self.W                # (N,Dout) @ (Dout,Din)

class ReLU:
    def forward(self, x):
        self.mask = (x > 0).to(x.dtype)
        return x * self.mask
    def backward(self, dout):
        return dout * self.mask

class Sigmoid:
    def forward(self, x):
        self.s = 1.0 / (1.0 + torch.exp(-x))
        return self.s
    def backward(self, dout):
        return dout * self.s * (1.0 - self.s)

class SoftmaxCrossEntropy:
    def forward(self, z, y):
        z = z - z.max(dim=1, keepdim=True).values                  # numerical stability
        logp = z - torch.log(torch.exp(z).sum(dim=1, keepdim=True))
        self.p, self.y = logp.exp(), y
        return -logp[torch.arange(len(y)), y].mean()
    def backward(self):
        dz = self.p.clone()
        dz[torch.arange(len(self.y)), self.y] -= 1.0               # p - onehot
        return dz / len(self.y)

class MLP:
    def __init__(self, sizes, act='relu', init=None, dtype=torch.float32):
        Act = ReLU if act == 'relu' else Sigmoid
        init = init or ('kaiming' if act == 'relu' else 'xavier')
        self.layers = []
        for i, (a, b) in enumerate(zip(sizes[:-1], sizes[1:])):
            self.layers.append(Linear(a, b, init, dtype))
            if i < len(sizes) - 2:                                  # no activation after the output layer
                self.layers.append(Act())
    def forward(self, x):
        for l in self.layers: x = l.forward(x)
        return x
    def backward(self, dout):
        for l in reversed(self.layers): dout = l.backward(dout)
        return dout
    def linears(self):
        return [l for l in self.layers if isinstance(l, Linear)]

# Sanity check: an untrained network should predict ~uniform probabilities -> loss ~ ln(10) = 2.303
set_seed()
mlp, crit = MLP([3072, 256, 10]), SoftmaxCrossEntropy()
loss = crit.forward(mlp.forward(X_tr[:256].flatten(1)), Y_tr[:256])
mlp.backward(crit.backward())
print(f'initial loss {loss.item():.3f}  (ln 10 = {math.log(10):.3f})')
print('gradient shapes:', [tuple(l.dW.shape) for l in mlp.linears()])

### 1(b) Gradient check: autograd and centred finite differences

* **Autograd reference:** the *same weights* are pushed through PyTorch's own ops (`F.linear`, `F.relu`, `F.cross_entropy`) and `torch.autograd.grad` gives the reference gradients.
* **Finite differences:** $\dfrac{\partial L}{\partial\theta_i}\approx\dfrac{L(\theta_i+h)-L(\theta_i-h)}{2h}$ for randomly sampled entries of every parameter tensor (doing all ~200k entries would need 400k forward passes).
* Relative error $= \lVert a-b\rVert/(\lVert a\rVert+\lVert b\rVert)$. The check is done in **float64**; the float32 version is shown afterwards for comparison.

In [ ]:
def manual_grads(mlp, crit, x, y):
    loss = crit.forward(mlp.forward(x), y)
    mlp.backward(crit.backward())
    return loss.item(), [t.clone() for l in mlp.linears() for t in (l.dW, l.db)]

def autograd_grads(mlp, x, y, act):
    # Independent reference: same weights, PyTorch ops + autograd
    params, h, lins = [], x, mlp.linears()
    for i, l in enumerate(lins):
        W, b = l.W.clone().requires_grad_(True), l.b.clone().requires_grad_(True)
        params += [W, b]
        h = F.linear(h, W, b)
        if i < len(lins) - 1:
            h = F.relu(h) if act == 'relu' else torch.sigmoid(h)
    loss = F.cross_entropy(h, y)
    return loss.item(), list(torch.autograd.grad(loss, params))

def finite_diff(mlp, crit, x, y, param, idxs, h):
    g = []
    for idx in idxs:
        old = param[idx].item()
        param[idx] = old + h; lp = crit.forward(mlp.forward(x), y).item()
        param[idx] = old - h; lm = crit.forward(mlp.forward(x), y).item()
        param[idx] = old
        g.append((lp - lm) / (2 * h))
    return torch.tensor(g, dtype=param.dtype, device=param.device)

def rel_err(a, b):
    return ((a - b).norm() / (a.norm() + b.norm()).clamp_min(1e-30)).item()

N_FD = 20                                          # sampled entries per parameter tensor
x64, y_gc = X_tr[:32].flatten(1).double(), Y_tr[:32]
rows = []
for act in ['relu', 'sigmoid']:
    set_seed(1)
    mlp, crit = MLP([3072, 64, 64, 10], act=act, dtype=torch.float64), SoftmaxCrossEntropy()
    l_man, g_man = manual_grads(mlp, crit, x64, y_gc)
    l_ag,  g_ag  = autograd_grads(mlp, x64, y_gc, act)
    params = [t for l in mlp.linears() for t in (l.W, l.b)]
    names  = [f'{p}{i + 1}' for i in range(3) for p in ('W', 'b')]
    gen = torch.Generator().manual_seed(0)
    for name, p, gm, ga in zip(names, params, g_man, g_ag):
        idxs = [tuple(torch.randint(0, s, (1,), generator=gen).item() for s in p.shape) for _ in range(N_FD)]
        fd = finite_diff(mlp, crit, x64, y_gc, p, idxs, h=1e-6)
        gm_s = torch.stack([gm[i] for i in idxs])
        rows.append(dict(activation=act, param=name, shape=str(tuple(p.shape)),
                         relerr_vs_autograd=rel_err(gm, ga), allclose_autograd=torch.allclose(gm, ga),
                         relerr_vs_finite_diff=rel_err(gm_s, fd),
                         allclose_finite_diff=torch.allclose(gm_s, fd, rtol=1e-5, atol=1e-8)))
    print(f'{act:8s} loss: manual {l_man:.10f} | autograd {l_ag:.10f}')

gc_table = pd.DataFrame(rows)
pd.set_option('display.float_format', '{:.2e}'.format)
display(gc_table)
pd.reset_option('display.float_format')

# Same finite-difference check in float32, to show why float64 is needed
set_seed(1)
mlp32, crit32 = MLP([3072, 64, 64, 10], 'relu', dtype=torch.float32), SoftmaxCrossEntropy()
_, g32 = manual_grads(mlp32, crit32, x64.float(), y_gc)
W1 = mlp32.linears()[0].W
gen = torch.Generator().manual_seed(0)
idxs = [tuple(torch.randint(0, s, (1,), generator=gen).item() for s in W1.shape) for _ in range(N_FD)]
gm_s = torch.stack([g32[0][i] for i in idxs])
for h in [1e-2, 1e-3, 1e-4, 1e-6]:
    fd = finite_diff(mlp32, crit32, x64.float(), y_gc, W1, idxs, h)
    print(f'float32, h={h:.0e}:  rel. error W1 = {rel_err(gm_s, fd):.2e}')

### 1(c) Training the MLP with SGD on flattened images
Architecture 3072 → 512 → 256 → 10 (ReLU), Kaiming initialization, plain mini-batch SGD with the update $\theta \leftarrow \theta - \eta\,\nabla_\theta L$ written by hand.

In [ ]:
LR_1C = 0.03

def sgd_step(mlp, lr):
    for l in mlp.linears():
        l.W -= lr * l.dW
        l.b -= lr * l.db

def accuracy_manual(mlp, X, Y, bs=1000):
    correct = 0
    for i in range(0, len(X), bs):
        correct += (mlp.forward(X[i:i + bs].flatten(1)).argmax(1) == Y[i:i + bs]).sum().item()
    return correct / len(X)

def run_1c():
    set_seed()
    mlp, crit = MLP([3072, 512, 256, 10]), SoftmaxCrossEntropy()
    hist = dict(iter_loss=[], train_loss=[], train_acc=[], val_acc=[])
    for ep in range(EPOCHS_1C):
        losses = []
        for xb, yb in batches(X_tr, Y_tr):
            loss = crit.forward(mlp.forward(xb.flatten(1)), yb)
            mlp.backward(crit.backward())
            sgd_step(mlp, LR_1C)
            losses.append(loss)
        losses = torch.stack(losses).tolist()
        hist['iter_loss'] += losses
        hist['train_loss'].append(float(np.mean(losses)))
        hist['train_acc'].append(accuracy_manual(mlp, X_tr[:5000], Y_tr[:5000]))
        hist['val_acc'].append(accuracy_manual(mlp, X_va, Y_va))
        print(f'epoch {ep + 1:2d}  loss {hist["train_loss"][-1]:.3f}  '
              f'train acc {hist["train_acc"][-1]:.3f}  val acc {hist["val_acc"][-1]:.3f}')
    return hist

h1c = cached('1c_mlp', run_1c)

def smooth(v, k=50):
    v = np.asarray(v, dtype=float)
    k = max(1, min(k, len(v)))
    return np.convolve(v, np.ones(k) / k, mode='valid')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
it = np.arange(len(h1c['iter_loss'])) / (len(h1c['iter_loss']) / len(h1c['train_loss']))
ax[0].plot(it, h1c['iter_loss'], alpha=0.25, lw=0.6, label='per iteration')
ax[0].plot(it[len(it) - len(smooth(h1c['iter_loss'])):], smooth(h1c['iter_loss']), label='moving average (50 it.)')
ax[0].set(xlabel='epoch', ylabel='training loss', title='1(c) Manual MLP: training loss'); ax[0].legend()
ep = np.arange(1, len(h1c['val_acc']) + 1)
ax[1].plot(ep, h1c['train_acc'], 'o-', label='train (5k subset)')
ax[1].plot(ep, h1c['val_acc'], 's-', label='validation')
ax[1].set(xlabel='epoch', ylabel='accuracy', title='1(c) Accuracy'); ax[1].legend()
plt.tight_layout(); savefig('1c_mlp'); plt.show()
print(f'final validation accuracy: {h1c["val_acc"][-1]:.4f}  (best {max(h1c["val_acc"]):.4f})')

### 1(d) Per-layer gradient norms at initialization (10-layer MLP)
10 linear layers (3072 → 256 × 9 → 10), one batch of 256 images, averaged over 5 random initializations.
Configurations: sigmoid + Xavier init, ReLU + Kaiming init, and (for comparison) ReLU + Xavier init.
The right panel shows the standard deviation of each layer's *input* (the forward signal), which helps explain the gradients.

In [ ]:
def layer_stats(act, init, seeds=5, width=256, depth=10):
    x, y = X_tr[:256].flatten(1), Y_tr[:256]
    gnorms, astds = [], []
    for s in range(seeds):
        set_seed(s)
        mlp, crit = MLP([3072] + [width] * (depth - 1) + [10], act=act, init=init), SoftmaxCrossEntropy()
        crit.forward(mlp.forward(x), y)
        mlp.backward(crit.backward())
        gnorms.append([l.dW.norm().item() for l in mlp.linears()])
        astds.append([l.x.std().item() for l in mlp.linears()])
    return np.mean(gnorms, 0), np.mean(astds, 0)

configs_1d = {'sigmoid + Xavier': ('sigmoid', 'xavier'),
              'ReLU + Kaiming':   ('relu', 'kaiming'),
              'ReLU + Xavier':    ('relu', 'xavier')}
stats_1d = {k: layer_stats(*v) for k, v in configs_1d.items()}

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
layers = np.arange(1, 11)
for k, (gn, sd) in stats_1d.items():
    ax[0].semilogy(layers, gn, 'o-', label=k)
    ax[1].semilogy(layers, sd, 'o-', label=k)
ax[0].set(xlabel='layer (1 = closest to the input)', ylabel=r'$\|\partial L/\partial W_l\|_F$',
          title='1(d) Gradient norm per layer at initialization', xticks=layers); ax[0].legend()
ax[1].set(xlabel='layer', ylabel='std of layer input', title='Forward signal: std of each layer input',
          xticks=layers); ax[1].legend()
plt.tight_layout(); savefig('1d_grad_norms'); plt.show()

display(pd.DataFrame({k: v[0] for k, v in stats_1d.items()}, index=[f'layer {i}' for i in layers]).T
        .style.format('{:.1e}'))
for k, (gn, _) in stats_1d.items():
    print(f'{k:18s} gradient norm ratio layer1/layer10 = {gn[0] / gn[-1]:.2e}')

#### ✍️ Interpretation of Task 1 (to write after running)
* **1(b):** what size are the relative errors vs autograd and vs finite differences, and why do the float32 finite differences depend so strongly on $h$ (truncation error vs round-off error)?
* **1(c):** final validation accuracy; is there a gap between train and validation; why can an MLP on raw pixels not go much further (no translation invariance, no weight sharing)?
* **1(d):** $\sigma'(x)\le 0.25$, so every sigmoid layer multiplies the backward signal by at most 0.25 → gradients shrink geometrically towards the input (**vanishing gradients**). ReLU passes gradients with factor 1 where active, and Kaiming init keeps the variance constant layer to layer, so the norms stay roughly flat. ReLU with Xavier shows that the *initialization* also matters: activations shrink by about $1/\sqrt2$ per layer.

## Task 2: Optimizers

### 2(a) SGD with momentum and Adam as `torch.optim.Optimizer` subclasses
* **SGD + momentum** (PyTorch convention): $v \leftarrow \mu v + g,\quad \theta \leftarrow \theta - \eta v$.
* **Adam:** $m \leftarrow \beta_1 m + (1-\beta_1)g$, $v \leftarrow \beta_2 v + (1-\beta_2)g^2$, bias correction $\hat m = m/(1-\beta_1^t)$, $\hat v = v/(1-\beta_2^t)$, update $\theta \leftarrow \theta - \eta\,\hat m/(\sqrt{\hat v}+\epsilon)$.
* Weight decay is the coupled L2 form $g \leftarrow g + \lambda\theta$, as in `torch.optim.SGD` / `torch.optim.Adam`.

In [ ]:
class MySGD(torch.optim.Optimizer):
    def __init__(self, params, lr=0.01, momentum=0.0, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, momentum=momentum, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad(): loss = closure()
        for group in self.param_groups:
            lr, mu, wd = group['lr'], group['momentum'], group['weight_decay']
            for p in group['params']:
                if p.grad is None: continue
                g = p.grad
                if wd != 0: g = g.add(p, alpha=wd)               # L2 penalty gradient
                if mu != 0:
                    state = self.state[p]
                    if 'v' not in state: state['v'] = torch.zeros_like(p)
                    v = state['v']
                    v.mul_(mu).add_(g)                           # v = mu*v + g
                    g = v
                p.add_(g, alpha=-lr)                             # theta = theta - lr*v
        return loss

class MyAdam(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad(): loss = closure()
        for group in self.param_groups:
            lr, (b1, b2), eps, wd = group['lr'], group['betas'], group['eps'], group['weight_decay']
            for p in group['params']:
                if p.grad is None: continue
                g = p.grad
                if wd != 0: g = g.add(p, alpha=wd)
                state = self.state[p]
                if not state:
                    state['t'] = 0
                    state['m'] = torch.zeros_like(p)
                    state['v'] = torch.zeros_like(p)
                state['t'] += 1
                t, m, v = state['t'], state['m'], state['v']
                m.mul_(b1).add_(g, alpha=1 - b1)                 # 1st moment
                v.mul_(b2).addcmul_(g, g, value=1 - b2)          # 2nd moment
                m_hat = m / (1 - b1 ** t)                        # bias correction
                v_hat = v / (1 - b2 ** t)
                p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)
        return loss

In [ ]:
# Equivalence test: identical model copies, identical batches, 100 steps, then compare all parameters.
# float64 on the CPU so that the only possible difference is the optimizer itself.
def opt_match_test(make_mine, make_ref, steps=100):
    set_seed(0)
    net = nn.Sequential(nn.Flatten(), nn.Linear(3072, 64), nn.ReLU(), nn.Linear(64, 10)).double()
    a, b = copy.deepcopy(net), copy.deepcopy(net)
    oa, ob = make_mine(a.parameters()), make_ref(b.parameters())
    Xc, Yc = X_tr[:2048].double().cpu(), Y_tr[:2048].cpu()
    for s in range(steps):
        i = (s * 64) % 2048
        xb, yb = Xc[i:i + 64], Yc[i:i + 64]
        for model, opt in ((a, oa), (b, ob)):
            opt.zero_grad()
            F.cross_entropy(model(xb), yb).backward()
            opt.step()
    pa, pb = list(a.parameters()), list(b.parameters())
    max_diff = max((x - y).abs().max().item() for x, y in zip(pa, pb))
    moved = max((x - y).abs().max().item() for x, y in zip(pa, net.parameters()))
    return dict(allclose=all(torch.allclose(x, y) for x, y in zip(pa, pb)), max_abs_diff=max_diff, max_param_change=moved)

tests = {
    'SGD lr=0.05':                (lambda p: MySGD(p, lr=0.05),                                  lambda p: torch.optim.SGD(p, lr=0.05)),
    'SGD+momentum 0.9':           (lambda p: MySGD(p, lr=0.01, momentum=0.9),                    lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9)),
    'SGD+momentum+wd 5e-4':       (lambda p: MySGD(p, lr=0.01, momentum=0.9, weight_decay=5e-4), lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9, weight_decay=5e-4)),
    'Adam lr=1e-3':               (lambda p: MyAdam(p, lr=1e-3),                                 lambda p: torch.optim.Adam(p, lr=1e-3)),
    'Adam betas=(0.8,0.99)+wd':   (lambda p: MyAdam(p, lr=3e-3, betas=(0.8, 0.99), weight_decay=1e-3),
                                   lambda p: torch.optim.Adam(p, lr=3e-3, betas=(0.8, 0.99), weight_decay=1e-3)),
}
match_table = pd.DataFrame({k: opt_match_test(*v) for k, v in tests.items()}).T
display(match_table)

### 2(b) Trajectories on $f(x,y)=\tfrac12(x^2+50y^2)$
The Hessian is $\mathrm{diag}(1, 50)$: condition number 50. Gradient descent is stable only for $\eta < 2/\lambda_{max} = 0.04$, and progress along $x$ is governed by $\lambda_{min}=1$. All three use the full (exact) gradient, starting from $(-9, 1.5)$.

In [ ]:
def f2(p): return 0.5 * (p[0] ** 2 + 50 * p[1] ** 2)

def trajectory(make_opt, start=(-9.0, 1.5), steps=60):
    p = torch.tensor(start, dtype=torch.float64, requires_grad=True)
    opt = make_opt([p])
    path = [p.detach().clone().numpy()]
    for _ in range(steps):
        opt.zero_grad(); f2(p).backward(); opt.step()
        path.append(p.detach().clone().numpy())
    return np.array(path)

traj_cfg = {'SGD (lr=0.035)':                  lambda p: MySGD(p, lr=0.035),
            'SGD + momentum (lr=0.01, μ=0.9)':  lambda p: MySGD(p, lr=0.01, momentum=0.9),
            'Adam (lr=0.5)':                    lambda p: MyAdam(p, lr=0.5)}
paths = {k: trajectory(v) for k, v in traj_cfg.items()}

xs, ys = np.meshgrid(np.linspace(-10, 10, 400), np.linspace(-2.2, 2.2, 400))
Z = 0.5 * (xs ** 2 + 50 * ys ** 2)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for a, (k, P) in zip(ax, paths.items()):
    a.contour(xs, ys, Z, levels=np.logspace(-1, 2.2, 14), cmap='Greys', linewidths=0.7)
    a.plot(P[:, 0], P[:, 1], 'o-', ms=2.5, lw=1)
    a.plot(*P[0], 'g^', ms=9, label='start'); a.plot(0, 0, 'r*', ms=12, label='minimum')
    a.set(title=k, xlabel='x', xlim=(-10, 10), ylim=(-2.2, 2.2))
ax[0].set_ylabel('y'); ax[0].legend(loc='lower right')
plt.tight_layout(); savefig('2b_trajectories'); plt.show()

plt.figure(figsize=(6, 3.8))
for k, P in paths.items():
    plt.semilogy([0.5 * (x ** 2 + 50 * y ** 2) + 1e-16 for x, y in P], label=k)
plt.xlabel('iteration'); plt.ylabel('f(x, y)'); plt.title('2(b) Objective vs iteration'); plt.legend()
plt.tight_layout(); savefig('2b_objective'); plt.show()

### Shared training utilities (used by Tasks 2(c), 3 and 4)
GPU-side data augmentation (flip, random crop with 4-pixel zero padding, **cutout**), **mixup**, an evaluation function,
and one training loop with mixed precision. *Train accuracy is always measured on clean (un-augmented) training images in eval mode*,
because accuracy on augmented or mixed-up batches is not comparable with validation accuracy.

In [ ]:
# ---------------- data augmentation on GPU tensors ----------------
def aug_flip(x):
    m = torch.rand(x.size(0), device=x.device) < 0.5
    return torch.where(m[:, None, None, None], x.flip(3), x)

def aug_crop(x, pad=4):
    B, C, H, W = x.shape
    xp = F.pad(x, (pad, pad, pad, pad))                          # zero = mean colour after normalization
    dy = torch.randint(0, 2 * pad + 1, (B,), device=x.device)
    dx = torch.randint(0, 2 * pad + 1, (B,), device=x.device)
    rows = (dy[:, None] + torch.arange(H, device=x.device))[:, None, :, None]   # (B,1,H,1)
    cols = (dx[:, None] + torch.arange(W, device=x.device))[:, None, None, :]   # (B,1,1,W)
    b = torch.arange(B, device=x.device)[:, None, None, None]
    c = torch.arange(C, device=x.device)[None, :, None, None]
    return xp[b, c, rows, cols]

def aug_cutout(x, size=16):
    # DeVries & Taylor (2017): zero a size x size square at a random centre (it may be clipped by the border)
    B, C, H, W = x.shape
    cy = torch.randint(0, H, (B, 1, 1), device=x.device)
    cx = torch.randint(0, W, (B, 1, 1), device=x.device)
    r = torch.arange(H, device=x.device)[None, :, None]
    c = torch.arange(W, device=x.device)[None, None, :]
    inside = (r >= cy - size // 2) & (r < cy + size // 2) & (c >= cx - size // 2) & (c < cx + size // 2)
    return x * (~inside)[:, None].to(x.dtype)

def make_aug(flip=False, crop=False, cutout=False):
    if not (flip or crop or cutout): return None
    def aug(x):
        if flip:   x = aug_flip(x)
        if crop:   x = aug_crop(x)
        if cutout: x = aug_cutout(x)
        return x
    return aug

def mixup(x, y, alpha):
    # Zhang et al. (2018): convex combination of two images and of their labels, lambda ~ Beta(alpha, alpha)
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

# ---------------- evaluation & training loop ----------------
@torch.no_grad()
def evaluate(model, X, Y, bs=1000):
    model.eval()
    loss_sum, correct = 0.0, 0
    for i in range(0, len(X), bs):
        xb, yb = X[i:i + bs].to(device), Y[i:i + bs].to(device)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
            out = model(xb)
        loss_sum += F.cross_entropy(out.float(), yb, reduction='sum').item()
        correct += (out.argmax(1) == yb).sum().item()
    return loss_sum / len(X), correct / len(X)

def train_model(model, opt, epochs, X=None, Y=None, *, aug=None, mixup_alpha=0.0, label_smoothing=0.0,
                sched=None, sched_per_iter=False, X_eval=None, Y_eval=None, print_every=5):
    X = X_tr if X is None else X
    Y = Y_tr if Y is None else Y
    X_eval = X[:5000] if X_eval is None else X_eval          # clean training images for train accuracy
    Y_eval = Y[:5000] if Y_eval is None else Y_eval
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    hist = {k: [] for k in ['iter_loss', 'train_loss', 'train_acc', 'val_loss', 'val_acc', 'lr']}
    hist['diverged'] = False
    for ep in range(epochs):
        model.train()
        losses = []
        for xb, yb in batches(X, Y):
            if aug is not None: xb = aug(xb)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
                if mixup_alpha > 0:
                    xb, ya, yb2, lam = mixup(xb, yb, mixup_alpha)
                    out = model(xb)
                    loss = lam * F.cross_entropy(out, ya, label_smoothing=label_smoothing) + \
                           (1 - lam) * F.cross_entropy(out, yb2, label_smoothing=label_smoothing)
                else:
                    loss = F.cross_entropy(model(xb), yb, label_smoothing=label_smoothing)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            if sched is not None and sched_per_iter: sched.step()
            losses.append(loss.detach().float())
        hist['lr'].append(opt.param_groups[0]['lr'])
        if sched is not None and not sched_per_iter: sched.step()
        losses = torch.stack(losses)
        hist['iter_loss'] += losses.tolist()
        hist['train_loss'].append(losses.mean().item())
        if not math.isfinite(hist['train_loss'][-1]):
            hist['diverged'] = True
            print(f'  diverged at epoch {ep + 1}')
            break
        hist['train_acc'].append(evaluate(model, X_eval, Y_eval)[1])
        vl, va = evaluate(model, X_va, Y_va)
        hist['val_loss'].append(vl); hist['val_acc'].append(va)
        if (ep + 1) % print_every == 0 or ep == 0 or ep == epochs - 1:
            print(f'  ep {ep + 1:3d}  loss {hist["train_loss"][-1]:.3f}  train acc {hist["train_acc"][-1]:.3f}  '
                  f'val acc {va:.3f}  lr {hist["lr"][-1]:.2e}')
    return hist

def n_params(model): return sum(p.numel() for p in model.parameters())

def small_cnn(bn=False, dropout=0.0):
    # 5 conv layers (32-32-64-64-128) + 2 fully connected layers
    def block(cin, cout):
        layers = [nn.Conv2d(cin, cout, 3, padding=1, bias=not bn)]
        if bn: layers.append(nn.BatchNorm2d(cout))
        return layers + [nn.ReLU(inplace=True)]
    net = nn.Sequential(*block(3, 32), *block(32, 32), nn.MaxPool2d(2),
                        *block(32, 64), *block(64, 64), nn.MaxPool2d(2),
                        *block(64, 128), nn.MaxPool2d(2),
                        nn.Flatten(), nn.Dropout(dropout), nn.Linear(128 * 4 * 4, 256), nn.ReLU(inplace=True),
                        nn.Dropout(dropout), nn.Linear(256, 10))
    for m in net.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None: nn.init.zeros_(m.bias)
    return net

print('small CNN parameters:', f'{n_params(small_cnn()):,}')

### 2(c) Small CNN with plain SGD, SGD + momentum and Adam at three learning rates each
Small CNN **without batch norm** (so the optimizer has to do the work), constant learning rate (no schedule, to isolate the optimizer), 15 epochs, batch 128, my own `MySGD` / `MyAdam`.
Learning-rate grids: SGD with momentum $\mu=0.9$ has an effective step about $\eta/(1-\mu) = 10\eta$, so its grid is roughly SGD's divided by 10.

In [ ]:
LRS_2C = {'SGD':            [0.01, 0.05, 0.2],
          'SGD + momentum': [0.001, 0.01, 0.05],
          'Adam':           [1e-4, 1e-3, 1e-2]}

def make_opt_2c(name, params, lr):
    if name == 'SGD':            return MySGD(params, lr=lr)
    if name == 'SGD + momentum': return MySGD(params, lr=lr, momentum=0.9)
    return MyAdam(params, lr=lr)

res_2c = {}
for name, lrs in LRS_2C.items():
    for lr in lrs:
        key = f'2c_{name.replace(" ", "").replace("+", "_")}_lr{lr:g}'
        def run(name=name, lr=lr):
            print(f'--- {name}, lr={lr:g}')
            set_seed()
            model = small_cnn(bn=False).to(device)
            return train_model(model, make_opt_2c(name, model.parameters(), lr), EPOCHS_2C)
        res_2c[(name, lr)] = cached(key, run)

In [ ]:
def best_val(h): return max(h['val_acc']) if h['val_acc'] else 0.1

rows = [dict(optimizer=n, lr=lr, final_train_loss=h['train_loss'][-1], best_val_acc=best_val(h),
             final_val_acc=h['val_acc'][-1] if h['val_acc'] else np.nan, diverged=h['diverged'],
             minutes=h['time_s'] / 60) for (n, lr), h in res_2c.items()]
table_2c = pd.DataFrame(rows)
display(table_2c.style.format({'lr': '{:g}', 'final_train_loss': '{:.3f}', 'best_val_acc': '{:.4f}',
                               'final_val_acc': '{:.4f}', 'minutes': '{:.1f}'}))

best_lr_2c = {n: max(lrs, key=lambda lr: best_val(res_2c[(n, lr)])) for n, lrs in LRS_2C.items()}
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for n, lrs in LRS_2C.items():
    accs = [best_val(res_2c[(n, lr)]) for lr in lrs]
    l, = ax[0].semilogx(lrs, accs, 'o-', label=n)
    for lr in lrs:
        if res_2c[(n, lr)]['diverged']:
            ax[0].plot(lr, best_val(res_2c[(n, lr)]), 'x', color=l.get_color(), ms=12, mew=2)
    h = res_2c[(n, best_lr_2c[n])]
    ep = np.arange(1, len(h['train_loss']) + 1)
    ax[1].plot(ep, h['train_loss'], 'o-', color=l.get_color(), label=f'{n} (lr={best_lr_2c[n]:g})')
    ax[2].plot(np.arange(1, len(h['val_acc']) + 1), h['val_acc'], 'o-', color=l.get_color(),
               label=f'{n} (lr={best_lr_2c[n]:g})')
ax[0].set(xlabel='learning rate (log scale)', ylabel='best validation accuracy',
          title='2(c) Validation accuracy vs learning rate (x = diverged)'); ax[0].legend()
ax[1].set(xlabel='epoch', ylabel='training loss', title='Training loss at the best learning rate'); ax[1].legend()
ax[2].set(xlabel='epoch', ylabel='validation accuracy', title='Validation accuracy at the best learning rate'); ax[2].legend()
plt.tight_layout(); savefig('2c_optimizers'); plt.show()

#### ✍️ Interpretation of Task 2 (to write after running)
* **2(a):** the custom optimizers match PyTorch to floating-point precision (state the max difference).
* **2(b):** plain SGD: with $\eta$ near $2/50$ the $y$-coordinate is multiplied by $(1-50\eta)$ each step, so it zig-zags across the narrow valley while $x$ shrinks only by $(1-\eta)$ per step, so progress along the valley floor is slow. Momentum averages the gradients: the oscillating $y$-components partly cancel and the consistent $x$-component accumulates, so it moves faster along the valley (with some overshoot). Adam divides each coordinate by its own RMS gradient, so both coordinates take steps of about $\eta$ regardless of curvature, which removes the effect of the bad conditioning; with a constant $\eta$ it then hovers around the minimum.
* **2(c):** compare how wide the "good" learning-rate range is for each optimizer (sensitivity), and which one converges fastest at its best rate.

## Task 3: Regularization and residual connections

### 3(a) Regularization on 5k training images
Small CNN **with batch norm**, trained on the first 5000 images of the training split (validation on the full 5k validation split).
SGD + momentum 0.9, lr 0.05 with cosine decay, 60 epochs. Settings: weight decay $5\times10^{-4}$, dropout 0.5 before both fully connected layers,
random horizontal flip + random crop (4-pixel padding), cutout (16×16), mixup ($\alpha=1$).

In [ ]:
N_SMALL = 512 if SMOKE_TEST else 5000
X_small, Y_small = X_tr[:N_SMALL], Y_tr[:N_SMALL]

CONFIGS_3A = {
    'baseline':      dict(),
    'weight decay':  dict(wd=5e-4),
    'dropout':       dict(dropout=0.5),
    'flip + crop':   dict(flip=True, crop=True),
    'cutout':        dict(cutout=True),
    'mixup':         dict(mixup=1.0),
    'all combined':  dict(wd=5e-4, dropout=0.5, flip=True, crop=True, cutout=True, mixup=1.0),
}

def run_3a(cfg):
    set_seed()
    model = small_cnn(bn=True, dropout=cfg.get('dropout', 0.0)).to(device)
    opt = MySGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=cfg.get('wd', 0.0))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_3A)
    aug = make_aug(cfg.get('flip', False), cfg.get('crop', False), cfg.get('cutout', False))
    return train_model(model, opt, EPOCHS_3A, X_small, Y_small, aug=aug, mixup_alpha=cfg.get('mixup', 0.0),
                       sched=sched, X_eval=X_small, Y_eval=Y_small, print_every=20)

res_3a = {}
for name, cfg in CONFIGS_3A.items():
    print(f'--- {name}')
    res_3a[name] = cached(f'3a_{name.replace(" ", "_").replace("+", "")}', lambda cfg=cfg: run_3a(cfg))

In [ ]:
table_3a = pd.DataFrame([dict(setting=n, train_acc=h['train_acc'][-1], val_acc=h['val_acc'][-1],
                              gap=h['train_acc'][-1] - h['val_acc'][-1], best_val_acc=max(h['val_acc']),
                              final_val_loss=h['val_loss'][-1]) for n, h in res_3a.items()])
display(table_3a.style.format({c: '{:.4f}' for c in table_3a.columns if c != 'setting'}))

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for n, h in res_3a.items():
    ep = np.arange(1, len(h['val_acc']) + 1)
    ls = '-' if n != 'all combined' else '--'
    ax[0].plot(ep, h['train_acc'], ls, label=n)
    ax[1].plot(ep, h['val_acc'], ls, label=n)
    ax[2].plot(ep, h['val_loss'], ls, label=n)
ax[0].set(xlabel='epoch', ylabel='accuracy', title='3(a) Train accuracy (clean images)')
ax[1].set(xlabel='epoch', ylabel='accuracy', title='Validation accuracy')
ax[2].set(xlabel='epoch', ylabel='loss', title='Validation loss', ylim=(0, 4))
ax[0].legend(fontsize=8)
plt.tight_layout(); savefig('3a_regularization'); plt.show()

### 3(b) Plain vs residual networks of depth $6n+2$ (8, 20, 56)
CIFAR-style ResNet (He et al., 2016, Sec. 4.2): 3×3 conv → three stages of $n$ basic blocks with 16/32/64 channels (stride-2 at the start of stages 2 and 3)
→ global average pooling → FC. Batch norm after every conv, Kaiming initialization.
The **only difference** is the shortcut: residual blocks compute $\mathrm{ReLU}(F(x) + x)$ with the parameter-free *option A* shortcut
(identity; when the shape changes, subsample by 2 and zero-pad the extra channels), plain blocks compute $\mathrm{ReLU}(F(x))$.
Both families therefore have **exactly the same parameters**.
Training: SGD momentum 0.9, lr 0.1 divided by 10 at 50% and 75% of training, weight decay $10^{-4}$, batch 128, flip + crop, 30 epochs.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride, shortcut):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut_type, self.stride, self.extra = shortcut, stride, cout - cin
        self.reshape = (stride != 1 or cin != cout)
        if shortcut == 'B' and self.reshape:          # projection shortcut (used in Task 4)
            self.proj = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def shortcut(self, x):
        if not self.reshape: return x                 # identity
        if self.shortcut_type == 'A':                 # subsample + zero-pad channels: no parameters
            x = x[:, :, ::self.stride, ::self.stride]
            return F.pad(x, (0, 0, 0, 0, self.extra // 2, self.extra - self.extra // 2))
        return self.proj(x)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.shortcut_type != 'none':
            out = out + self.shortcut(x)              # the ONLY difference between plain and residual
        return F.relu(out)

class CifarResNet(nn.Module):
    # depth = 6n + 2;  shortcut: 'A' (parameter-free), 'B' (1x1 projection), 'none' (plain network)
    def __init__(self, n, shortcut='A', widths=(16, 32, 64), num_classes=10):
        super().__init__()
        self.conv = nn.Conv2d(3, widths[0], 3, 1, 1, bias=False); self.bn = nn.BatchNorm2d(widths[0])
        blocks, cin = [], widths[0]
        for i, w in enumerate(widths):
            for j in range(n):
                blocks.append(BasicBlock(cin, w, 2 if (i > 0 and j == 0) else 1, shortcut)); cin = w
        self.blocks = nn.Sequential(*blocks)
        self.fc = nn.Linear(cin, num_classes)
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        x = F.relu(self.bn(self.conv(x)))
        x = self.blocks(x)
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

DEPTHS = [8, 20, 56]
display(pd.DataFrame({d: {'n': (d - 2) // 6,
                          'plain params': n_params(CifarResNet((d - 2) // 6, 'none')),
                          'residual params': n_params(CifarResNet((d - 2) // 6, 'A'))} for d in DEPTHS}).T)

In [ ]:
def run_3b(depth, kind):
    set_seed()
    model = CifarResNet((depth - 2) // 6, shortcut='A' if kind == 'residual' else 'none').to(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[int(0.5 * EPOCHS_3B), int(0.75 * EPOCHS_3B)], gamma=0.1)
    return train_model(model, opt, EPOCHS_3B, aug=make_aug(flip=True, crop=True), sched=sched, print_every=5)

res_3b = {}
for depth in DEPTHS:                       # deepest last: if time runs out, depths 8 and 20 are already done
    for kind in ['plain', 'residual']:
        print(f'--- {kind}-{depth}')
        res_3b[(kind, depth)] = cached(f'3b_{kind}{depth}', lambda d=depth, k=kind: run_3b(d, k))

In [ ]:
colors = {8: 'tab:blue', 20: 'tab:orange', 56: 'tab:red'}
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
for (kind, d), h in res_3b.items():
    it_loss = h['iter_loss']
    x = np.arange(len(it_loss)) / (len(it_loss) / len(h['train_loss']))
    s = smooth(it_loss, 100)
    a = ax[0] if kind == 'plain' else ax[1]
    a.plot(x[len(x) - len(s):], s, color=colors[d], label=f'{kind}-{d}')
    ax[2].plot(np.arange(1, len(h['val_acc']) + 1), h['val_acc'], '--' if kind == 'plain' else '-',
               color=colors[d], label=f'{kind}-{d}')
ylim = (0, max(max(h['train_loss'][:2]) for h in res_3b.values()) * 1.05)
ax[0].set(title='3(b) Plain networks: training loss', xlabel='epoch', ylabel='training loss (100-it. moving avg)', ylim=ylim)
ax[1].set(title='Residual networks: training loss', xlabel='epoch', ylabel='training loss (100-it. moving avg)', ylim=ylim)
ax[2].set(title='Validation accuracy (dashed = plain)', xlabel='epoch', ylabel='accuracy')
for a in ax: a.legend()
plt.tight_layout(); savefig('3b_plain_vs_residual'); plt.show()

table_3b = pd.DataFrame([dict(model=f'{k}-{d}', params=n_params(CifarResNet((d - 2) // 6, 'A' if k == 'residual' else 'none')),
                              final_train_loss=h['train_loss'][-1], train_acc=h['train_acc'][-1],
                              val_acc=h['val_acc'][-1], minutes=h['time_s'] / 60)
                         for (k, d), h in sorted(res_3b.items(), key=lambda t: (t[0][1], t[0][0]))])
display(table_3b.style.format({'final_train_loss': '{:.4f}', 'train_acc': '{:.4f}', 'val_acc': '{:.4f}',
                               'minutes': '{:.1f}', 'params': '{:,}'}))

#### ✍️ Interpretation of Task 3 (to write after running)
* **3(a):** with only 5k images the baseline memorizes the training set (train acc → ~100%, validation loss rises after a few epochs). Which regularizers shrink the train/val gap most, which ones actually raise validation accuracy, and why is train accuracy *lower* for the data-augmentation methods (they make the training task harder)? Does "all combined" over-regularize in 60 epochs?
* **3(b):** degradation problem: the plain net gets **worse in training loss** as it gets deeper (plain-56 above plain-20). This is an *optimization* problem, not overfitting (the training error itself is higher). A deeper net could copy the shallower one and set the extra layers to identity, yet SGD does not find that solution. With shortcuts each block only has to learn a residual $F(x)$, identity is the default ($F=0$), and gradients reach early layers through the identity paths, so deeper residual nets train to *lower* loss.

## Task 4 (optional): Best CNN with ≤ 500k parameters and ≤ 20 epochs
ResNet-14-style network (n = 2 blocks per stage) with widths 24/48/96 and projection (option B) shortcuts.
Training: SGD with Nesterov momentum 0.9, one-cycle learning-rate schedule (max lr 0.1), weight decay $5\times10^{-4}$, label smoothing 0.1,
flip + crop + cutout, batch 128, 20 epochs. Model selection is done on the validation split only; the test set is used once, at the end.

In [ ]:
W4, N4 = (24, 48, 96), 2
BASE_4 = dict(shortcut='B', flip=True, crop=True, cutout=True, wd=5e-4, ls=0.1, lr=0.1)
ABLATIONS_4 = {
    'full model':             {},
    'no shortcut (plain)':    dict(shortcut='none'),
    'no cutout':              dict(cutout=False),
    'no augmentation':        dict(flip=False, crop=False, cutout=False),
    'no weight decay':        dict(wd=0.0),
    'no label smoothing':     dict(ls=0.0),
}
p4 = n_params(CifarResNet(N4, 'B', W4))
assert p4 <= 500_000
print(f'Task 4 model parameters: {p4:,}  (plain variant: {n_params(CifarResNet(N4, "none", W4)):,})')

def run_4(cfg, key):
    set_seed()
    model = CifarResNet(N4, cfg['shortcut'], W4).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9, nesterov=True, weight_decay=cfg['wd'])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=cfg['lr'], epochs=EPOCHS_4,
                                                steps_per_epoch=math.ceil(len(X_tr) / BS))
    h = train_model(model, opt, EPOCHS_4, aug=make_aug(cfg['flip'], cfg['crop'], cfg['cutout']),
                    label_smoothing=cfg['ls'], sched=sched, sched_per_iter=True, print_every=5)
    torch.save(model.state_dict(), f'{RESULTS}/{key}.pt')
    return h

res_4 = {}
if RUN_TASK4:
    for name, change in ABLATIONS_4.items():
        cfg, key = {**BASE_4, **change}, '4_' + name.replace(' ', '_').replace('(', '').replace(')', '')
        print(f'--- {name}')
        res_4[name] = cached(key, lambda cfg=cfg, key=key: run_4(cfg, key))

In [ ]:
if RUN_TASK4:
    table_4 = pd.DataFrame([dict(setting=n, final_val_acc=h['val_acc'][-1],
                                 delta_vs_full=h['val_acc'][-1] - res_4['full model']['val_acc'][-1],
                                 final_train_acc=h['train_acc'][-1], minutes=h['time_s'] / 60)
                            for n, h in res_4.items()])
    display(table_4.style.format({'final_val_acc': '{:.4f}', 'delta_vs_full': '{:+.4f}',
                                  'final_train_acc': '{:.4f}', 'minutes': '{:.1f}'}))

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    for n, h in res_4.items():
        ep = np.arange(1, len(h['val_acc']) + 1)
        ax[0].plot(ep, h['val_acc'], 'o-' if n == 'full model' else '-', ms=3, label=n)
        ax[1].plot(ep, h['train_loss'], 'o-' if n == 'full model' else '-', ms=3, label=n)
    ax[0].set(xlabel='epoch', ylabel='validation accuracy', title='Task 4 ablation: validation accuracy')
    ax[1].set(xlabel='epoch', ylabel='training loss', title='Training loss')
    ax[0].legend(fontsize=8)
    plt.tight_layout(); savefig('4_ablation'); plt.show()

In [ ]:
# ---- Final model = best configuration on the VALIDATION split; test set evaluated ONCE ----
if RUN_TASK4:
    best_4 = max(res_4, key=lambda n: res_4[n]['val_acc'][-1])
    def test_once():
        cfg = {**BASE_4, **ABLATIONS_4[best_4]}
        key = '4_' + best_4.replace(' ', '_').replace('(', '').replace(')', '')
        model = CifarResNet(N4, cfg['shortcut'], W4).to(device)
        model.load_state_dict(torch.load(f'{RESULTS}/{key}.pt', map_location=device))
        test_loss, test_acc = evaluate(model, X_te, Y_te)
        return dict(model=best_4, params=n_params(model), val_acc=res_4[best_4]['val_acc'][-1],
                    test_acc=test_acc, test_loss=test_loss)
    test_result = cached('4_TEST_RESULT', test_once)     # cached: re-running the notebook never re-touches the test set
    print(f"Final model: '{test_result['model']}'  ({test_result['params']:,} parameters, {EPOCHS_4} epochs)")
    print(f"validation accuracy {test_result['val_acc']:.4f}  |  TEST accuracy {test_result['test_acc']:.4f}")

#### ✍️ Interpretation of Task 4 (to write after running)
Which design choices matter most under a 20-epoch budget (largest drop when removed)? Is the shortcut still important at this depth (14 layers)?
Is the test accuracy close to the validation accuracy (no over-tuning to the validation split)?

## Summary of key numbers

In [ ]:
print(f"1(c) manual MLP          : val acc {h1c['val_acc'][-1]:.4f}")
for n in LRS_2C:
    print(f"2(c) {n:15s}     : best lr {best_lr_2c[n]:g}, best val acc {best_val(res_2c[(n, best_lr_2c[n])]):.4f}")
for n, h in res_3a.items():
    print(f"3(a) {n:15s}     : train {h['train_acc'][-1]:.3f}  val {h['val_acc'][-1]:.3f}  gap {h['train_acc'][-1] - h['val_acc'][-1]:+.3f}")
for (k, d), h in sorted(res_3b.items(), key=lambda t: (t[0][1], t[0][0])):
    print(f"3(b) {k + '-' + str(d):15s}     : final train loss {h['train_loss'][-1]:.4f}  val acc {h['val_acc'][-1]:.4f}")
if RUN_TASK4:
    print(f"4    final model         : test acc {test_result['test_acc']:.4f}")

## References
* K. He, X. Zhang, S. Ren, J. Sun, *Deep Residual Learning for Image Recognition*, CVPR 2016.
* D. P. Kingma, J. Ba, *Adam: A Method for Stochastic Optimization*, ICLR 2015.
* T. DeVries, G. W. Taylor, *Improved Regularization of Convolutional Neural Networks with Cutout*, arXiv:1708.04552, 2017.
* H. Zhang, M. Cisse, Y. N. Dauphin, D. Lopez-Paz, *mixup: Beyond Empirical Risk Minimization*, ICLR 2018.
* L. N. Smith, N. Topin, *Super-Convergence: Very Fast Training of Neural Networks Using Large Learning Rates*, 2019.